# Token Streaming with Redis Streams

This notebook demonstrates using `RedisStreamTransport` for reliable token streaming.

**Features:**
- Reliable event delivery via Redis Streams
- **Consumer groups** for multiple clients (XREADGROUP/XACK/XCLAIM)
- Event replay capability
- Automatic stream trimming
- Pending message tracking and recovery

**Event Types:**
- `token` - Individual tokens from LLM
- `tool_call` - Tool invocation
- `tool_result` - Tool response
- `complete` - Stream complete

## Setup

In [1]:
import os
import asyncio
from dotenv import load_dotenv

load_dotenv()

REDIS_URL = os.getenv("REDIS_URL", "redis://redis:6379")
print(f"Redis URL: {REDIS_URL}")

Redis URL: redis://redis:6379


## Create Stream Transport

In [2]:
from redis_openai_agents import RedisStreamTransport

# Create a stream for agent output
stream = RedisStreamTransport(
    stream_name="agent_output",
    redis_url=REDIS_URL,
    consumer_group="demo_clients",
)

print(f"Stream created: {stream.stream_name}")
print(f"Consumer group: {stream.consumer_group}")

Stream created: agent_output
Consumer group: demo_clients


## Publish Events

In [3]:
# Simulate streaming tokens from an agent
tokens = ["Hello", ",", " ", "how", " ", "can", " ", "I", " ", "help", "?"]

print("Publishing token events...")
for token in tokens:
    msg_id = stream.publish(
        event_type="token",
        data={"token": token}
    )
    print(f"  Published: '{token}' -> {msg_id}")

# Publish completion event
stream.publish(
    event_type="complete",
    data={"reason": "end_of_response"}
)
print("\nStream complete event published")

Publishing token events...
  Published: 'Hello' -> 1768415721469-0
  Published: ',' -> 1768415721469-1
  Published: ' ' -> 1768415721469-2
  Published: 'how' -> 1768415721470-0
  Published: ' ' -> 1768415721470-1
  Published: 'can' -> 1768415721470-2
  Published: ' ' -> 1768415721470-3
  Published: 'I' -> 1768415721470-4
  Published: ' ' -> 1768415721470-5
  Published: 'help' -> 1768415721470-6
  Published: '?' -> 1768415721470-7

Stream complete event published


## Consume Events

In [4]:
# Consume events from the beginning
print("Consuming events from stream...\n")

events = stream.read_all(count=20)
for event in events:
    print(f"Event: {event['type']}")
    print(f"  Data: {event['data']}")
    print(f"  ID: {event['id']}")
    print()

Consuming events from stream...

Event: token
  Data: {'token': 'Hello'}
  ID: 1768415721469-0

Event: token
  Data: {'token': ','}
  ID: 1768415721469-1

Event: token
  Data: {'token': ' '}
  ID: 1768415721469-2

Event: token
  Data: {'token': 'how'}
  ID: 1768415721470-0

Event: token
  Data: {'token': ' '}
  ID: 1768415721470-1

Event: token
  Data: {'token': 'can'}
  ID: 1768415721470-2

Event: token
  Data: {'token': ' '}
  ID: 1768415721470-3

Event: token
  Data: {'token': 'I'}
  ID: 1768415721470-4

Event: token
  Data: {'token': ' '}
  ID: 1768415721470-5

Event: token
  Data: {'token': 'help'}
  ID: 1768415721470-6

Event: token
  Data: {'token': '?'}
  ID: 1768415721470-7

Event: complete
  Data: {'reason': 'end_of_response'}
  ID: 1768415721470-8



## Stream Info

In [5]:
info = stream.info()
print(f"Stream length: {info['length']}")
print(f"Consumer groups: {info['groups']}")

Stream length: 12
Consumer groups: 1


## Consumer Groups

Consumer groups enable reliable delivery to multiple clients.
Each message is delivered to only one consumer in the group.

In [6]:
# Create a new stream for consumer group demo
cg_stream = RedisStreamTransport(
    stream_name="consumer_group_demo",
    redis_url=REDIS_URL,
    consumer_group="demo_group",
)

# Publish some events
for i in range(4):
    cg_stream.publish(event_type="token", data={"token": f"word{i}"})

print(f"Published 4 events to {cg_stream.stream_name}")

Published 4 events to consumer_group_demo


In [7]:
# Consumer 1 reads first 2 events
events1 = cg_stream.read_group(consumer="consumer1", count=2)
print(f"Consumer1 received {len(events1)} events:")
for e in events1:
    print(f"  - {e['data']}")

# Consumer 2 reads next 2 events (different messages!)
events2 = cg_stream.read_group(consumer="consumer2", count=2)
print(f"\nConsumer2 received {len(events2)} events:")
for e in events2:
    print(f"  - {e['data']}")

Consumer1 received 2 events:
  - {'token': 'word0'}
  - {'token': 'word1'}

Consumer2 received 2 events:
  - {'token': 'word2'}
  - {'token': 'word3'}


## Pending Messages and ACK

Read messages are "pending" until acknowledged. This enables reliable delivery.

In [8]:
# Check pending messages (unacknowledged)
pending = cg_stream.pending()
print(f"Pending messages: {pending['count']}")
print(f"Consumers with pending: {pending['consumers']}")

# Acknowledge messages from consumer1
ids_to_ack = [e['id'] for e in events1]
acked = cg_stream.ack(ids_to_ack)
print(f"\nAcknowledged {acked} messages from consumer1")

# Check pending again
pending_after = cg_stream.pending()
print(f"Pending after ACK: {pending_after['count']}")

Pending messages: 4
Consumers with pending: {'consumer1': 2, 'consumer2': 2}

Acknowledged 2 messages from consumer1
Pending after ACK: 2


## Claim Pending Messages

If a consumer dies, another can claim its pending messages.

In [9]:
# Consumer2's messages are still pending - claim them as consumer3
ids_to_claim = [e['id'] for e in events2]
claimed = cg_stream.claim(
    consumer="consumer3",
    min_idle_ms=0,  # Claim immediately (normally wait for timeout)
    ids=ids_to_claim
)

print(f"Consumer3 claimed {len(claimed)} messages from consumer2:")
for e in claimed:
    print(f"  - {e['data']}")

# Now ACK them as consumer3
cg_stream.ack([e['id'] for e in claimed])
print(f"\nAll messages now acknowledged")
print(f"Final pending: {cg_stream.pending()['count']}")

# Cleanup
cg_stream.delete()

Consumer3 claimed 2 messages from consumer2:
  - {'token': 'word2'}
  - {'token': 'word3'}

All messages now acknowledged
Final pending: 0


## Cleanup

In [10]:
stream.delete()
print("Stream deleted")

Stream deleted


## Summary

This notebook demonstrated:

1. **Stream Creation** - Create Redis Stream with consumer group
2. **Event Publishing** - Publish token and completion events
3. **Event Consumption** - Read events from the stream with `read_all()`
4. **Consumer Groups** - Multiple consumers with `read_group()`
5. **Message ACK** - Acknowledge processed messages with `ack()`
6. **Pending Messages** - Track unacknowledged messages with `pending()`
7. **Claim Messages** - Recover from dead consumers with `claim()`

**Use Cases:**
- Real-time token streaming to multiple clients
- Event replay for debugging
- Reliable message delivery with acknowledgment
- Fault-tolerant streaming with message recovery